<!-- GENERATED from diff-hist/docs/public/fleet_and_egress_load.md by tools/gen_public_docs.py — edit the source, not here. -->

# Fleet and Egress Load

Egress volume and traffic value estimates across the global M-Lab server fleet — This is an
internal dashboard intended to support fleet lifecycle management and capacity planning.

A world map plus a filterable inventory table let you see where traffic and
serving value are concentrated across the fleet.

For information about Differential Histograms and how they expose anomalies in Internet mid-paths
see the **[project overview](https://annealing.mattmathis.net/differential-histograms/)**.

See **[complete](https://annealing.mattmathis.net/differential-histograms/fleet_and_egress_load)** Fleet and Egress Load documentation.

In [ ]:
# --- Setup ---
import os, sys, json
from datetime import datetime, date, time, timedelta, timezone

# Locate the repo root (directory containing `converter/`) regardless of where
# Voila/Jupyter is launched from.
_root = os.path.abspath(os.getcwd())
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, "converter")):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

import ipywidgets as widgets
import plotly.graph_objects as go
import pandas as pd
from IPython.display import display, HTML, Markdown

from converter import query_builder as qb, runtime as rt
from converter.widget_builder import Controls

client = rt.bq_client()

# Dashboard variable metadata baked in at conversion time.
VARIABLES = json.loads(r"""
[
  {
    "name": "costModel",
    "type": "custom",
    "label": "Value Model",
    "description": "Regional per SKU costs used to generate overall costs",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "Flat $0.10 per Gigabyte", "value": "Flat $0.10 per Gigabyte" }
    ],
    "current": { "value": "Flat $0.10 per Gigabyte" },
    "query_sql": "Flat $0.10 per Gigabyte"
  },
  {
    "name": "duration",
    "type": "custom",
    "label": "",
    "description": "Number of days to include in the sampel",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "1", "value": "1" },
      { "text": "7", "value": "7" },
      { "text": "30", "value": "30" }
    ],
    "current": { "value": "1" },
    "query_sql": "1, 7, 30"
  },
  {
    "name": "endDate",
    "type": "textbox",
    "label": "",
    "description": "Sample end",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "2026-04-20", "value": "2026-04-20" }
    ],
    "current": { "value": "2026-04-20" },
    "query_sql": "2026-04-20",
    "dynamic_default": "(date.today() - timedelta(days=2)).isoformat()"
  },
  {
    "name": "display",
    "type": "custom",
    "label": "Level of detail",
    "description": "Select between site details and site summaries.",
    "hide": 0,
    "multi": true,
    "options": [
      { "text": "summaries", "value": "summaries" },
      { "text": "metros", "value": "metros" },
      { "text": "sites", "value": "sites" }
    ],
    "current": {
      "value": [
        "metros"
      ]
    },
    "query_sql": "summaries, metros, sites"
  }
]
""")


In [ ]:
# --- URL parameter presets (webapp mode) ---
# Voila injects the request query string into os.environ["QUERY_STRING"] before
# executing the notebook.  get_query_string() also handles the preheat-kernel
# case (blocks until the request arrives).  Falls back gracefully in plain
# Jupyter where neither is set.
# For scripted or test overrides, set DASH_PRESETS to a JSON object.
import urllib.parse

url_params = {}
try:
    from voila.utils import get_query_string
    _qs = get_query_string() or ""
    for _k, _vs in urllib.parse.parse_qs(_qs).items():
        url_params[_k] = _vs[0] if len(_vs) == 1 else _vs
except Exception:
    pass

_env = os.environ.get("DASH_PRESETS")
if _env:
    url_params.update(json.loads(_env))

# sites= and ISPs= param handling.
# sites= pre-selects servers by site code (e.g. sites=lga04,lga05).
# ISPs= pre-selects ISPs by AS number (e.g. ISPs=7922,8030).
# If sites= is present but anchor= is not, derive anchor from first site code.
_sites_param = [s.strip() for s in url_params.get('sites', '').split(',') if s.strip()]
_isp_asns    = [s.strip() for s in url_params.get('ISPs',  '').split(',') if s.strip()]
if _sites_param and 'anchor' not in url_params:
    url_params['anchor'] = _sites_param[0][:3]
if _sites_param:
    url_params['region'] = _sites_param


In [ ]:
# --- Dashboard controls (dropdowns; query-backed ones are chained) ---

w_from = w_to = w_duration = None

# Date row: start/end range (exp) or end + duration (otherwise). Empty when the
# flavor has no date pickers.
if w_from is not None:
    _date_row = widgets.HBox([w_from, w_to], layout=widgets.Layout(margin='2px 0'))
elif w_to is not None and w_duration is not None:
    _date_row = widgets.HBox([w_to, w_duration], layout=widgets.Layout(margin='2px 0'))
else:
    _date_row = widgets.HTML('')

# The method (or methodsrc, in exp) selector drives date-picker visibility; the
# date row is spliced into the controls column right after it.
_method_var = 'methodsrc' if 'methodsrc' in [v['name'] for v in VARIABLES] else 'method'
# Put endDate + duration on one row (duration second) where both exist (fleet).
_var_names = [v['name'] for v in VARIABLES]
_hgroups = [['endDate', 'duration']] if 'endDate' in _var_names and 'duration' in _var_names else []
ctrl = Controls(VARIABLES, client, presets=url_params,
                asn_presets={'ClientISP': _isp_asns} if _isp_asns else None,
                after={_method_var: _date_row}, hgroups=_hgroups)
w_run = widgets.Button(description="Run / Refresh", button_style="primary", icon="play")
_date_label = widgets.HTML('')   # filled from query results after Run

# Hide the date row when the backend token is 'cached'; show it otherwise.
_method_w = ctrl.widgets.get(_method_var)
def _toggle_date_row(*_):
    _is_cached = str(getattr(_method_w, 'value', '')).split('-')[0] == 'cached'
    _date_row.layout.display = 'none' if _is_cached else ''
if _method_w is not None:
    _method_w.observe(_toggle_date_row, names='value')
_toggle_date_row()

# "Extra rows" (if present) is shown only when the selected servers span more
# than one metro (distinct 3-letter IATA prefixes of the site codes).
_extra_w   = ctrl.widgets.get('extra_rows')
_servers_w = ctrl.widgets.get('region')
def _toggle_extra_rows(*_):
    _sel = _servers_w.value if _servers_w is not None else ()
    _metros = {str(s)[:3] for s in _sel}
    _row = getattr(_extra_w, 'widget', _extra_w)
    _row.layout.display = '' if len(_metros) > 1 else 'none'
if _extra_w is not None and _servers_w is not None:
    _servers_w.observe(_toggle_extra_rows, names='value')
    _toggle_extra_rows()


In [ ]:
# --- Panels (converted from the dashboard) ---
SUMMARY_PANELS  = [
  {
    'id': 1,
    'title': 'Server egress value estimated from TCPinfo using ${costModel}  for ${duration} days ending ${endDate}',
    'type': 'table',
    'sql': r"""
SELECT -- reorder fields to make it prettier
  * EXCEPT (lat, long),
  lat, long
FROM `mm_preproduction.global_fleet_inventory`(DATE_SUB("$endDate", INTERVAL ${duration:raw}-1 DAY), DATE("$endDate"))
WHERE
  ("${display}" like '%summar%' AND tag LIKE '!%') OR
  ("${display}" like '%metro%' AND level LIKE '%Metro%') OR
  ("${display}" like '%site%' AND level LIKE '%Site%')
""".strip(),
    'layout': {},
    'skip_if_field_none': False,
  },
]
METRIC_LAYOUTS  = json.loads(r"""{
  "MeanThroughputMbps": {
    "type": "log",
    "autorange": false,
    "range": [
      -0.3,
      3.3
    ],
    "gridcolor": "#333"
  },
  "MinRTT": {
    "type": "log",
    "autorange": false,
    "range": [
      -0.3,
      3.0
    ],
    "gridcolor": "#333"
  },
  "linearMinRTT": {
    "type": "linear",
    "autorange": false,
    "range": [
      0,
      300
    ],
    "gridcolor": "#333"
  },
  "LossRate": {
    "type": "log",
    "autorange": true,
    "gridcolor": "#333"
  }
}""")
REPEAT_VAR      = None

out = widgets.Output()


def _diagnostics(ctx):
    rows = [(k, ", ".join(v) if isinstance(v, list) else str(v))
            for k, v in ctx.items()]
    return pd.DataFrame(rows, columns=["variable", "value"])


def render(_=None):
    ctx = ctrl.context()
    to_dt   = (datetime.combine(w_to.value, time(), tzinfo=timezone.utc)
               if w_to and w_to.value else datetime.now(timezone.utc))
    if w_from is not None:
        from_dt = (datetime.combine(w_from.value, time(), tzinfo=timezone.utc)
                   if w_from.value else to_dt - timedelta(days=7))
    else:
        from_dt = to_dt - timedelta(days=(w_duration.value if w_duration else 7))
    cache = {}

    def query(sql):
        if sql not in cache:
            cache[sql] = rt.run_query(client, sql)
        return cache[sql]

    out.clear_output(wait=True)
    with out:
        # Default to "Summary" when table_style is absent (e.g. fleet dashboard).
        table_style = ctx.get("table_style",
                               "Summary" if SUMMARY_PANELS else "none")
        if table_style != "none":
            _sctx = dict(ctx)
            if "table_style" in ctx:
                # Prod: map table_style → verbose flag expected by regional_report SQL.
                _sctx["verbose"] = "true" if table_style == "Verbose" else "false"
            # Other flavors (barchart, fleet) pass verbose directly from ctx.
            for p in SUMMARY_PANELS:
                display(Markdown("### " + qb.interpolate(p["title"], ctx)))
                sql = qb.interpolate(p["sql"], _sctx, from_dt=from_dt, to_dt=to_dt)
                try:
                    _df = query(sql)
                except Exception as exc:
                    display(HTML(f"<pre>query failed: {exc}</pre>"))
                    _df = None
                if _df is not None:
                    if p.get("type") == "barchart":
                        import ipywidgets as _ipyw
                        _link = _ipyw.HTML(
                            value='<p style="color:var(--jp-content-font-color1,#212121);font-size:12px">'
                                  '&#8592; click a bar to open Regional Details</p>'
                        )
                        _fw = rt.metro_barchart_clickable(
                            _df, isp_count=ctx.get("ISPcount", "5"),
                            link_widget=_link)
                        _fw._config = {"responsive": False}
                        display(_fw)
                        display(_link)
                    else:
                        _display_sel = ctx.get("display") or []
                        if isinstance(_display_sel, str): _display_sel = [_display_sel]
                        if any(d in _display_sel for d in ("metros", "sites")):
                            _map_df = _df[_df["lat"].notna() & _df["long"].notna()].copy()
                            if not _map_df.empty:
                                _fw = go.FigureWidget(rt.fleet_map(_map_df))
                                _fw._config = {"responsive": True}   # scale to page width
                                display(_fw)
                        display(HTML(
                            '<div style="height:500px;overflow:auto">'
                            + rt.to_html_sticky(_df, index=False, na_rep="")
                            + '</div>'
                        ))

        repeats = ctx.get(REPEAT_VAR) or []
        if isinstance(repeats, str):
            repeats = [repeats]

        selected_metrics = ctx.get("metrics") or []
        if isinstance(selected_metrics, str):
            selected_metrics = [selected_metrics]

        # One Python call per selected metric fetches data for all client ISPs.
        bulk_by_metric = {}
        _site_regex = qb.format_regex(ctx.get("region") or [])
        _isp_regex  = rt.asn_regex(repeats)
        # "Extra rows" pads the BQ row count only when the selected servers span
        # more than one metro; it is not used for the ISP selection or display.
        _servers = ctx.get("region") or []
        if isinstance(_servers, str):
            _servers = [_servers]
        _metros = {s[:3] for s in _servers}
        _extra_rows = int(ctx.get("extra_rows") or 0) if len(_metros) > 1 else 0
        _isp_count = int(ctx.get("ISPcount", 10)) + _extra_rows
        for metric in selected_metrics:
            try:
                bulk_by_metric[metric] = rt.fetch_histograms(
                    client,
                    method=ctx.get("method", "cached"),
                    field=metric,
                    site_regex=_site_regex,
                    isp_count=_isp_count,
                    bin_size=int(ctx.get("binSize", 50)),
                    x_axis=ctx.get("xAxis", "none"),
                    from_dt=from_dt,
                    to_dt=to_dt,
                    isp_regex=_isp_regex,
                    dataset=ctx.get("dataset",
                                    "mlab-collaboration.mm_preproduction"),
                )
            except Exception as exc:
                bulk_by_metric[metric] = exc

        # Update cached date range label from metroStart/metroEnd in query results.
        for _mdf in bulk_by_metric.values():
            if isinstance(_mdf, pd.DataFrame) and 'metroStart' in _mdf.columns:
                _s = pd.to_datetime(_mdf['metroStart'].dropna().min()).date()
                _e = pd.to_datetime(_mdf['metroEnd'].dropna().max()).date()
                _date_label.value = (
                    '<div style="font-size:12px;color:grey;margin:2px 0">'
                    '<b>Cached data:</b> ' + str(_s) + ' – ' + str(_e) + '</div>')
                break

        for value in repeats:
            asn = str(value).split()[0]
            display(HTML(f"<h3>{REPEAT_VAR}: {value}</h3>"))
            figs = []
            for metric in selected_metrics:
                df_all = bulk_by_metric.get(metric)
                if isinstance(df_all, Exception):
                    figs.append(widgets.HTML(f"<b>{metric}</b><pre>{df_all}</pre>"))
                    continue
                df = df_all[df_all["ISPname"].str.startswith(asn + " ")]
                try:
                    fig = rt.plotly_combined_figure(
                        df, {"xaxis": METRIC_LAYOUTS.get(metric, {})}, title=metric,
                        sites=_servers)
                    figs.append(go.FigureWidget(fig))
                except Exception as exc:
                    figs.append(widgets.HTML(f"<b>{metric}</b><pre>{exc}</pre>"))
            if figs:
                display(widgets.HBox(figs, layout=widgets.Layout(flex_flow="row wrap")))

        _diag_out = widgets.Output()
        with _diag_out:
            display(_diagnostics(ctx))
        _diag_acc = widgets.Accordion(children=[_diag_out])
        _diag_acc.set_title(0, 'Selector Diagnostics')
        _diag_acc.selected_index = None   # collapsed by default
        display(_diag_acc)


w_run.on_click(render)
if url_params:
    render()


In [ ]:
# --- Display the app ---
# _date_row is inserted inside ctrl.box (right after the method selector) by
# Controls(after=...); only the status label, Run button, and output remain here.
display(widgets.VBox([ctrl.box, w_run, out, _date_label]))
